In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import numpy as np
import evaluate
from transformers import TrainingArguments, Trainer
import torch
import pandas as pd
from datasets import Dataset
from transformers import DataCollatorWithPadding

/Users/aditisuresh/FakeNewsDetection/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# Set the path to the file you'd like to load
fake_path = "Data/Fake.csv"
real_path = "Data/True.csv"
tweets_df_train_path = "Data/train.csv"
tweets_df_test_path = "Data/test.csv"

fake_df = pd.read_csv(fake_path)
fake_df['label'] = "fake"
real_df = pd.read_csv(real_path)
real_df['label'] = "real"

real_df['all_text'] = real_df['title'] + " " + real_df['text']
fake_df['all_text'] = fake_df['title'] + " " + fake_df['text']

df = pd.concat([fake_df, real_df], ignore_index=True)

label_mapping = {'real': 0, 'fake': 1}
df['label'] = df['label'].map(label_mapping)

# 2. Handle missing text
df['all_text'] = df['all_text'].fillna("").astype(str)

# 3. CRITICAL: Rename 'label' to 'labels' so the Trainer doesn't drop it
df.rename(columns={'label': 'labels'}, inplace=True)

In [ ]:
model_id = "answerdotai/ModernBERT-base"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)

def tokenize_function(examples):
    return tokenizer(
        examples["all_text"], 
        truncation=True, 
        max_length=1024
    )

hf_dataset = Dataset.from_pandas(df)

# 2. Apply the tokenization
# (batched=True is crucial here so it processes chunks of text at once)
tokenized_datasets = hf_dataset.map(tokenize_function, batched=True)

split_datasets = tokenized_datasets.train_test_split(test_size=0.2, seed=42)

Loading weights: 100%|██████████| 136/136 [00:00<00:00, 5112.80it/s]
ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Map: 100%|██████████| 44898/44898 [00:15<00:00, 2927.64 examples/s]


In [ ]:
print(df['labels'].unique())

KeyError: 'label'

In [ ]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="./modernbert-fake-news",
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    bf16=True # Keep True if using an Ampere/newer GPU
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Update the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split_datasets["train"],
    eval_dataset=split_datasets["test"],
    compute_metrics=compute_metrics,
    data_collator=data_collator, # <-- ADDED HERE!
)

In [10]:
trainer.train()

/Users/aditisuresh/FakeNewsDetection/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
article_text = "Breaking: Scientists have just discovered that the moon is entirely made of cheese."

# Prepare the text
inputs = tokenizer(article_text, return_tensors="pt").to(model.device)

# Get predictions
with torch.no_grad():
    outputs = model(**inputs)
    
prediction = outputs.logits.argmax(dim=-1).item()

# Map the output back to a human-readable label
if prediction == 1:
    print("Classification: Fake News")
else:
    print("Classification: Real News")